In [0]:
# ── Silver: dim_competitions ─────────────────────────────────────
from pyspark.sql import functions as F

bronze = spark.table("football_catalog.bronze.competitions")

silver = (bronze
    .select(
        F.col("competition_id").cast("string"),
        F.trim(F.col("name")).alias("competition_name"),
        F.trim(F.col("sub_type")).alias("competition_type"),
        F.trim(F.col("country_name")).alias("country_name"),
        F.col("confederation").cast("string"),
        F.current_timestamp().alias("_silver_ts")
    )
    .dropDuplicates(["competition_id"])
    .filter(F.col("competition_id").isNotNull())
)

# Quality check
nulls = silver.filter(F.col("competition_id").isNull()).count()
print(f"Null PKs (should be 0): {nulls}")

(silver.write.format("delta").mode("overwrite")
 .option("overwriteSchema","true")
 .saveAsTable("football_catalog.silver.dim_competitions"))

spark.table("football_catalog.silver.dim_competitions").show(5)
